# Lab 4.3 &mdash; Multi-Tool Orchestration

**Level:** Advanced &nbsp;|&nbsp; **Est. time:** 35 min &nbsp;|&nbsp; **Day 2 &middot; Module 4 &mdash; Tool Calling &amp; MCP**

### What you'll do
- Pull the argument out of the request &mdash; and notice when there is not one
- Thread one tool's output into the next tool's argument, which is what &lsquo;multi-tool&rsquo; means
- Recognise a repeated call, because that is what a stuck agent looks like from outside
- Stop on a budget, and report <em>why</em> you stopped

> **How this lab works.** Fill every `BLANK`, then run the **Self-check** cell under each section.
> Graded cells are plain Python and never call a model, so your score never depends on a
> live endpoint. Cells marked **Run it for real** do call the sandbox model; if it is not
> reachable they print how to fix it instead of crashing.

> **Builds on Lab 4.1's contract.** A tool that returns instead of raising is what makes
> a multi-step run recoverable; here you find out what still goes wrong when it does.

In [ ]:
# ---------------------------------------------------------------- Setup: run me first
import os, json, time, textwrap
from typing import Any, Callable

WORK = os.path.join("/tmp", "awmas-lab-4-03")
os.makedirs(WORK, exist_ok=True)

# ---- self-check plumbing -------------------------------------------------
_results = []

def check(name: str, fn: Callable[[], Any], hint: str = "") -> None:
    """[PASS] / [FAIL] / [TODO] for one assertion. An unfilled blank prints [TODO]."""
    try:
        ok = bool(fn())
    except NameError:
        print(f"[TODO] {name}")
        _results.append(None)
        return
    except Exception as exc:
        print(f"[FAIL] {name} -- {type(exc).__name__}: {exc}")
        _results.append(False)
        return
    print(("[PASS] " if ok else "[FAIL] ") + name + ("" if ok else (" -- " + hint if hint else "")))
    _results.append(ok)

def guard(fn: Callable[[], Any], default: Any = None) -> Any:
    """Run fn(). If a blank above is still unfilled, say so and carry on -- never crash Run All."""
    try:
        return fn()
    except NameError:
        print("(a blank above is still unfilled -- fill it in, then re-run this cell)")
        return default

def score() -> None:
    done = [r for r in _results if r is not None]
    passed = sum(1 for r in done if r)
    todo = sum(1 for r in _results if r is None)
    print(f"\nScore: {passed}/{len(_results)}" + (f"   ({todo} still TODO)" if todo else ""))

# ---- the sandbox model ---------------------------------------------------
# Your sandbox already has an LLM configured -- nothing to install, no key to register.
# These two values are read from the environment so this notebook never hardcodes an endpoint.
LLM_BASE_URL = (os.environ.get("LAB_LLM_BASE_URL") or os.environ.get("OPENAI_BASE_URL")
                or os.environ.get("LITELLM_BASE_URL"))
LLM_MODEL    = (os.environ.get("LAB_LLM_MODEL") or os.environ.get("OPENAI_MODEL")
                or os.environ.get("LITELLM_MODEL"))
LLM_API_KEY  = os.environ.get("OPENAI_API_KEY", "sandbox")

def llm_ready() -> bool:
    if not LLM_BASE_URL or not LLM_MODEL:
        print("Model not configured. In a sandbox terminal run `env | grep -i llm` and set:")
        print("  export LAB_LLM_BASE_URL=...    # the gateway URL from your welcome sheet")
        print("  export LAB_LLM_MODEL=...       # the model name from your welcome sheet")
        return False
    return True

_llm = None
def get_llm(temperature: float = 0.0):
    """A LangChain chat model pointed at the sandbox gateway (OpenAI-compatible)."""
    global _llm
    if _llm is None:
        from langchain_openai import ChatOpenAI
        _llm = ChatOpenAI(model=LLM_MODEL, base_url=LLM_BASE_URL,
                          api_key=LLM_API_KEY, temperature=temperature)
    return _llm

def ask(prompt: str, system: str | None = None) -> str:
    """One stateless call. Returns text, or an error string -- never raises."""
    try:
        msgs = ([("system", system)] if system else []) + [("human", prompt)]
        return get_llm().invoke(msgs).content
    except Exception as exc:
        return f"<model unavailable: {type(exc).__name__}: {exc}>"

print("work dir:", WORK)
print("model   :", LLM_MODEL or "(not configured -- graded cells still work)")

In [ ]:
# ------------------------------------------------- the case file (synthetic, self-contained)
# One domain runs through all five Module 4 labs -- the same payment exceptions as Day 1,
# now reached through tools the agent chooses, and then through tools it did not write.
# Nothing here is real data and nothing leaves this notebook.

LEDGER = {
    "PMT-1001": {"amount": 250000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "settled",  "value_date": "2026-09-01", "reason_code": None},
    "PMT-1002": {"amount":  48250.75, "ccy": "EUR", "counterparty": "ACME-EU",
                 "status": "failed",   "value_date": "2026-09-02", "reason_code": "INSUFFICIENT_FUNDS"},
    "PMT-1003": {"amount": 990000.00, "ccy": "USD", "counterparty": "ZENITH",
                 "status": "held",     "value_date": "2026-09-02", "reason_code": "LIMIT_BREACH"},
    "PMT-1004": {"amount":   1200.00, "ccy": "GBP", "counterparty": "ACME-UK",
                 "status": "failed",   "value_date": "2026-09-03", "reason_code": "INVALID_IBAN"},
    "PMT-1005": {"amount": 750000.00, "ccy": "USD", "counterparty": "NORTHWIND",
                 "status": "held",     "value_date": "2026-09-03", "reason_code": "SANCTIONS_REVIEW"},
}

POLICY = {
    "INSUFFICIENT_FUNDS": "Retry once after 24h. If it fails again, notify the client desk. No manual funding.",
    "LIMIT_BREACH":       "Payments above USD 500,000 need Treasury approval before release.",
    "INVALID_IBAN":       "Return to originator with code R04. Never repair beneficiary details in-house.",
    "SANCTIONS_REVIEW":   "Hold. Compliance decides. Operations must not release or cancel.",
}

# Which reason codes may an agent resolve on its own, and which need a human?
NEEDS_HUMAN = {"LIMIT_BREACH", "SANCTIONS_REVIEW"}

print(f"{len(LEDGER)} payments, {len(POLICY)} policy rules loaded")

In [ ]:
# ------------------------------------------------- carried forward from Lab 1.2 of Module 1
# The tools you wrote in Lab 1.2 of Module 1. Nothing to fill in -- they are here so this
# notebook runs on its own. Note the docstrings: they name the case AND the boundary.

def lookup_payment(ref: str) -> str:
    """Return the ledger record for one payment reference such as 'PMT-1002'.

    Use when you need the status, amount, counterparty or reason code of a specific payment.
    Not for searching across payments.
    """
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, **record})


def policy_for(reason_code: str) -> str:
    """Return the operating policy for one failure reason code, e.g. 'LIMIT_BREACH'.

    Use after you know why a payment failed and need to know what to do about it.
    """
    return POLICY.get(reason_code, f"no policy on file for reason code {reason_code!r}")


TOOLS = {"lookup_payment": lookup_payment, "policy_for": policy_for}
print("carried forward:", ", ".join(TOOLS))

In [ ]:
# ------------------------------------------------- the toolkit these labs route between
# Four tools over the same ledger. Two read, one explains, one moves money -- which is the
# distinction that matters once an agent is choosing between them on its own.

def search_payments(counterparty: str = "", status: str = "") -> str:
    """Return the ledger records whose counterparty or status matches a query."""
    hits = [{"ref": r, **v} for r, v in LEDGER.items()
            if (not counterparty or v["counterparty"] == counterparty)
            and (not status or v["status"] == status)]
    return json.dumps(hits)


def release_payment(ref: str) -> str:
    """Release one held payment so that it settles."""
    record = LEDGER.get(ref)
    if record is None:
        return f"no payment found with reference {ref!r}"
    return json.dumps({"ref": ref, "released": True, "was": record["status"]})


TOOLKIT_FNS = {"lookup_payment": lookup_payment, "search_payments": search_payments,
               "policy_for": policy_for, "release_payment": release_payment}
print("toolkit:", ", ".join(TOOLKIT_FNS))

## Concept

&ldquo;Multi-tool&rdquo; is three separate problems wearing one name:

1. **Selection** &mdash; which tool. Lab 4.2 measured this.
2. **Arguments** &mdash; what to pass, extracted from a request written by a human.
3. **Sequencing** &mdash; the interesting one. `policy_for` needs a reason code that only
   `lookup_payment` can supply, so step two's argument does not exist until step one has run.

Plus the failure that ends production agents: a loop. Two calls with the same tool and the same
arguments cannot produce a different answer, so the second one is always wasted &mdash; and the
tenth one is an incident.

## Section 1 &mdash; The argument is in the request

Before any tool runs, something has to turn *&ldquo;why did PMT-1004 fail?&rdquo;* into `ref="PMT-1004"`.
Note the second half of the job: recognising when the request names no reference at all.

In [ ]:
import re

REF_RE = re.compile(r"\bPMT-\d{4}\b")

def extract_ref(request: str):
    """The payment reference this request names, or None if it names none."""
    m = REF_RE.search(request or "")
    return m.group(0) if m else None

In [ ]:
# --- Self-check: Section 1
check("a reference mid-sentence is found",
      lambda: extract_ref("Why did PMT-1004 fail?") == "PMT-1004")
check("and one at the end of a sentence, punctuation and all",
      lambda: extract_ref("Please pull up PMT-1001.") == "PMT-1001")
check("a request that names no payment returns None, not a guess",
      lambda: extract_ref("Which payments involve NORTHWIND?") is None,
      "inventing a plausible reference here is how an agent reads the wrong account")
check("a too-short number is not a reference",
      lambda: extract_ref("ticket PMT-99 is open") is None)
check("empty input is handled",
      lambda: extract_ref("") is None)

## Section 2 &mdash; Step two's argument comes from step one

A plan is a list of steps whose arguments may be **references** rather than values:

- `"$request.ref"` &mdash; the reference extracted from what the user asked
- `"$0.reason_code"` &mdash; the `reason_code` field of step 0's result

Resolving those references at run time is the whole mechanism behind a multi-step tool agent.

In [ ]:
PLANS = {
    "explain_failure": [
        {"tool": "lookup_payment", "args": {"ref": "$request.ref"}},
        {"tool": "policy_for",     "args": {"reason_code": "$0.reason_code"}},
    ],
    "find_for_counterparty": [
        {"tool": "search_payments", "args": {"counterparty": "NORTHWIND"}},
    ],
}

def resolve_arg(value, request_ref, results):
    """Resolve one argument that may point at the request or at an earlier step's result."""
    if not isinstance(value, str) or not value.startswith("$"):
        return value
    if value == "$request.ref":
        return request_ref
    step, _, field = value[1:].partition(".")
    try:
        payload = json.loads(results[int(step)])
    except (ValueError, TypeError):
        # That step did not return a record -- there is no field to thread forward. Carrying
        # the gap forward as None beats raising: the next tool can still report a real failure.
        return None
    return payload.get(field)


def run_plan(plan, request, tools=None):
    """Run every step in order, resolving each argument just before the call."""
    tools = TOOLKIT_FNS if tools is None else tools
    ref, results, trace = extract_ref(request), [], []
    for step in plan:
        args = {k: resolve_arg(v, ref, results) for k, v in step["args"].items()}
        results.append(tools[step["tool"]](**args))
        trace.append((step["tool"], args))
    return {"results": results, "trace": trace}

In [ ]:
# --- Self-check: Section 2
check("step 0 is called with the reference from the request",
      lambda: run_plan(PLANS["explain_failure"], "Why did PMT-1002 fail?")["trace"][0]
              == ("lookup_payment", {"ref": "PMT-1002"}))
check("step 1's argument came from step 0's OUTPUT, not from the request",
      lambda: run_plan(PLANS["explain_failure"], "Why did PMT-1002 fail?")["trace"][1][1]
              == {"reason_code": "INSUFFICIENT_FUNDS"},
      "that is the whole point of sequencing -- the argument did not exist until step 0 returned")
check("and the plan ends with the policy text for that code",
      lambda: "Retry once after 24h" in
              run_plan(PLANS["explain_failure"], "Why did PMT-1002 fail?")["results"][1])
check("a different payment threads a different code through",
      lambda: run_plan(PLANS["explain_failure"], "What about PMT-1003?")["trace"][1][1]
              == {"reason_code": "LIMIT_BREACH"})
check("a literal argument is passed through untouched",
      lambda: run_plan(PLANS["find_for_counterparty"], "anything")["trace"][0][1]
              == {"counterparty": "NORTHWIND"})
check("a request naming no reference does not crash the plan",
      lambda: isinstance(run_plan(PLANS["explain_failure"], "no reference here")["results"][0], str),
      "Lab 4.1's contract is what keeps this recoverable instead of fatal")

def _show():
    out = run_plan(PLANS["explain_failure"], "Why did PMT-1004 fail?")
    for tool, args in out["trace"]:
        print(f"  {tool:18} {args}")
    print("  ->", out["results"][-1][:80])
guard(_show)

## Section 3 &mdash; The stop conditions

Two calls with the same tool and the same arguments return the same answer. So the second one
buys nothing, and an agent that keeps making it is stuck &mdash; not slow.

A budget catches the runs a loop check misses: no repeat, just a plan that will not end.

In [ ]:
def call_key(tool: str, args: dict):
    """An identity for one call, so that a repeat of it is recognisable."""
    return (tool, json.dumps(args, sort_keys=True, default=str))


def run_guarded(plan, request, budget=4, tools=None):
    """Run a plan, refusing to repeat an identical call and stopping at the budget.

    Always returns an outcome -- 'completed', 'loop' or 'budget' -- and the trace so far.
    """
    tools = TOOLKIT_FNS if tools is None else tools
    ref, results, trace, seen = extract_ref(request), [], [], set()
    for step in plan:
        if len(trace) >= budget:
            return {"outcome": "budget", "trace": trace, "results": results,
                    "why": f"stopped after {budget} calls without finishing"}
        args = {k: resolve_arg(v, ref, results) for k, v in step["args"].items()}
        key = call_key(step["tool"], args)
        if key in seen:
            return {"outcome": "loop", "trace": trace, "results": results,
                    "why": f"{step['tool']} was already called with these arguments"}
        seen.add(key)
        results.append(tools[step["tool"]](**args))
        trace.append((step["tool"], args))
    return {"outcome": "completed", "trace": trace, "results": results, "why": ""}

In [ ]:
# --- Self-check: Section 3
_repeat = [{"tool": "lookup_payment", "args": {"ref": "$request.ref"}},
           {"tool": "lookup_payment", "args": {"ref": "$request.ref"}}]
_long   = [{"tool": "lookup_payment",  "args": {"ref": "PMT-1001"}},
           {"tool": "lookup_payment",  "args": {"ref": "PMT-1002"}},
           {"tool": "lookup_payment",  "args": {"ref": "PMT-1003"}},
           {"tool": "lookup_payment",  "args": {"ref": "PMT-1004"}},
           {"tool": "lookup_payment",  "args": {"ref": "PMT-1005"}}]

check("two identical calls are the same key",
      lambda: call_key("lookup_payment", {"ref": "PMT-1002"})
              == call_key("lookup_payment", {"ref": "PMT-1002"}))
check("argument order does not make a call look new",
      lambda: call_key("search_payments", {"counterparty": "ZENITH", "status": "held"})
              == call_key("search_payments", {"status": "held", "counterparty": "ZENITH"}),
      "otherwise the same call reordered slips past the loop check")
check("a different argument is a different call",
      lambda: call_key("lookup_payment", {"ref": "PMT-1002"})
              != call_key("lookup_payment", {"ref": "PMT-1003"}))
check("a different tool is a different call",
      lambda: call_key("lookup_payment", {"ref": "PMT-1002"})
              != call_key("release_payment", {"ref": "PMT-1002"}))

check("a clean plan completes",
      lambda: run_guarded(PLANS["explain_failure"], "Why did PMT-1002 fail?")["outcome"] == "completed")
check("a repeated call stops the run",
      lambda: run_guarded(_repeat, "About PMT-1002")["outcome"] == "loop")
check("and the reason names the tool that repeated",
      lambda: "lookup_payment" in run_guarded(_repeat, "About PMT-1002")["why"])
check("the repeated call was NOT executed",
      lambda: len(run_guarded(_repeat, "About PMT-1002")["trace"]) == 1)
check("a plan longer than the budget stops at the budget",
      lambda: run_guarded(_long, "x", budget=3)["outcome"] == "budget")
check("and it stopped after exactly the budget, not one call later",
      lambda: len(run_guarded(_long, "x", budget=3)["trace"]) == 3)
check("every outcome carries a trace, including the failures",
      lambda: all("trace" in run_guarded(p, "About PMT-1002", budget=3)
                  for p in (_repeat, _long, PLANS["explain_failure"])),
      "a run you cannot see is a run you cannot debug -- Module 7 builds on this")

## Section 4 &mdash; The whole thing, on a small suite

Four requests, four expectations. This is the shape of Lab 4.2's harness applied to *behaviour*
rather than to selection &mdash; and it is the last thing you build before Day 3 makes it the gate.

In [ ]:
SUITE = [
    ("Why did PMT-1002 fail?",       "explain_failure",        "completed"),
    ("What about PMT-1003?",         "explain_failure",        "completed"),
    ("Which ones are NORTHWIND's?",  "find_for_counterparty",  "completed"),
    ("Why did nothing-here fail?",   "explain_failure",        "completed"),
]

def run_suite() -> list:
    """Run each case and report its outcome and step count."""
    rows = []
    for request, plan_name, expected in SUITE:
        out = run_guarded(PLANS[plan_name], request)
        rows.append((request, out["outcome"], len(out["trace"]), out["outcome"] == expected))
    return rows

def _report():
    print(f"{'request':32}{'outcome':12}{'steps':>6}  ok")
    print("-" * 60)
    for request, outcome, steps, good in run_suite():
        print(f"{request[:30]:32}{outcome:12}{steps:>6}  {'yes' if good else 'NO'}")
guard(_report)

In [ ]:
# --- Self-check: Section 4
check("every case in the suite reaches its expected outcome",
      lambda: all(row[3] for row in run_suite()))
check("the two-step plan really took two steps",
      lambda: run_suite()[0][2] == 2)
check("the one-step plan took one",
      lambda: run_suite()[2][2] == 1)
check("a request with no reference still completes rather than crashing",
      lambda: run_suite()[3][1] == "completed",
      "it completes with a useless answer -- which is a different bug, and one the agent can see")

## Run it for real

Let the model choose the plan instead of you. Notice what it does with the fourth request, the
one naming no payment: the honest answer is to ask a question, and nothing in this design lets it.

In [ ]:
if llm_ready():
    def _pick():
        catalogue = "\n".join(f"- {n}: {[s['tool'] for s in p]}" for n, p in PLANS.items())
        for request, _, _ in SUITE:
            reply = ask(f"Plans available:\n{catalogue}\n\nRequest: {request}\n\n"
                        "Reply with one plan name, or the word NEITHER.",
                        system="Reply with a single word and nothing else.")
            print(f"  {request[:34]:36} -> {reply.strip()[:40]}")
    guard(_pick)

### Read it

If the model answers `explain_failure` for the request that names no payment, the plan will run,
`lookup_payment` will return &ldquo;no payment found&rdquo;, and `policy_for` will be handed `None`.
Every step succeeded and the answer is worthless.

That is failure 4 from the deck &mdash; the one that looks like success &mdash; and no amount of loop
detection catches it. Module 5 gives it a home: a supervisor whose job includes deciding that
*neither* plan applies.

In [ ]:
score()

## Your turn

1. Add a `NEITHER` outcome to `run_guarded` for a request the plans do not cover. What should the
   agent return to the user, and how is that different from an error?
2. `resolve_arg` returns `None` when an earlier step returned prose instead of a record, and the
   run then completes with a worthless answer. Rewrite it to return one of Lab 4.1's structured
   failures instead, and decide who should stop the run: the resolver, the plan, or the tool.
3. The loop check compares exact arguments. An agent that retries `PMT-1002`, then `PMT-1003`, then
   `PMT-1002` again defeats it. Widen the check to a repeat *within a window* and see what it costs
   in false positives.